In [9]:
from langgraph.graph import StateGraph, START, MessagesState, END
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langchain_core.messages.utils import trim_messages, count_tokens_approximately

In [10]:
load_dotenv()

True

In [11]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GOOGLE_GENAI_KEY"),
)

In [12]:
MAX_TOKENS = 150

In [20]:
def call_node(state: MessagesState):
    messages = trim_messages(
        state["messages"],
        strategy="last",
        token_counter = count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print("current token count:", count_tokens_approximately(messages))
    for message in messages:
        print(message.content)

    response = llm.invoke(messages)

    return {
        "messages": [response]
    }

    

In [21]:
builder = StateGraph(MessagesState)
builder.add_node("call_node", call_node)

builder.add_edge(START, "call_node")
builder.add_edge("call_node", END)

In [22]:
checkpoint = InMemorySaver()
graph = builder.compile(checkpointer=checkpoint)

In [29]:
config = {
    "configurable": {
        "thread_id": "1"
    }
}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name "}]},
    config=config
)

result["messages"][-1].content


current token count: 8
What is my name 


[{'type': 'text',
  'text': 'I don’t know your name. As an AI, I don’t have access to your personal information, identity, or private files unless you have previously shared that information with me in our current conversation.',
  'extras': {'signature': 'EnEKbwERTTIPVOsahJWgcoOtgdILjc1XhV+pnxOhHPQkHdEgVk8MNrq9yxU6sIDAJ/c4zgKd9eJvGLW0ifsKzcnwFdYYFMEz8MvH64/GlPCgcdfSvwC/cSwWntljD86Np6LqnYNy9AhrJKUA/SsHnpzAYw=='}}]